In [2]:
# Data Exploration Notebook

This notebook explores the BioHub Cell Tracking dataset, including:
- Loading Zarr v3 samples
- Visualizing 3D cell stacks
- Understanding data structure and statistics
- Examining frame sequences

SyntaxError: invalid decimal literal (1390735186.py, line 5)

In [4]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from biohub_tracking.data.zarr_loader import iter_frames
from biohub_tracking.segmentation.segmenter import CellSegmenter
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set up visualization parameters
plt.rcParams['figure.figsize'] = (12, 8)
np.set_printoptions(precision=3, suppress=True)

ModuleNotFoundError: No module named 'biohub_tracking'

## 1. Load Sample Data

First, let's check if sample data exists and load it.

In [5]:
# Define data paths
data_dir = Path("../data")
train_dir = data_dir / "train"
test_dir = data_dir / "test"

# Check if data directories exist
print(f"Data directory exists: {data_dir.exists()}")
print(f"Train directory exists: {train_dir.exists()}")
print(f"Test directory exists: {test_dir.exists()}")

# List available samples
if train_dir.exists():
    train_samples = sorted(train_dir.glob("*.zarr"))
    print(f"\nTrain samples: {len(train_samples)}")
    for sample in train_samples[:5]:  # Show first 5
        print(f"  - {sample.name}")
        
if test_dir.exists():
    test_samples = sorted(test_dir.glob("*.zarr"))
    print(f"\nTest samples: {len(test_samples)}")
    for sample in test_samples[:5]:  # Show first 5
        print(f"  - {sample.name}")

Data directory exists: False
Train directory exists: False
Test directory exists: False


## 2. Explore Frame Structure

Load a sample and examine the frame structure and statistics.

In [ ]:
# Load and inspect a sample if available
sample_data = []
sample_path = None

if train_dir.exists():
    samples = sorted(train_dir.glob("*.zarr"))
    if samples:
        sample_path = samples[0]
        print(f"Loading sample: {sample_path.name}")
        
        # Load frames
        frame_count = 0
        shape_info = {}
        
        for frame_index, image in iter_frames(sample_path):
            sample_data.append((frame_index, image))
            frame_count += 1
            shape_info[frame_index] = image.shape
            
            if frame_count <= 3:  # Print first 3 frames
                print(f"  Frame {frame_index}: shape={image.shape}, dtype={image.dtype}, "
                      f"min={image.min():.1f}, max={image.max():.1f}, mean={image.mean():.1f}")
        
        print(f"\nTotal frames loaded: {frame_count}")
        print(f"Frame shape (Z, Y, X): {shape_info.get(0, 'N/A')}")
elif test_dir.exists():
    samples = sorted(test_dir.glob("*.zarr"))
    if samples:
        sample_path = samples[0]
        print(f"Loading sample from test: {sample_path.name}")
        # Same loading code
else:
    print("No data directory found. Please ensure data is in ../data/train or ../data/test")

## 3. Visualize Frames

Display sample frames from the sequence.

In [ ]:
if sample_data:
    # Visualize first 3 frames with middle Z slice
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, (frame_idx, (frame_index, image)) in enumerate(zip(range(3), sample_data[:3])):
        # Get middle Z slice
        z_middle = image.shape[0] // 2
        img_slice = image[z_middle, :, :]
        
        # Plot with proper scaling
        im = axes[idx].imshow(img_slice, cmap='viridis')
        axes[idx].set_title(f'Frame {frame_index} (Z={z_middle}/{image.shape[0]-1})')
        axes[idx].set_xlabel('X pixels')
        axes[idx].set_ylabel('Y pixels')
        plt.colorbar(im, ax=axes[idx])
    
    plt.tight_layout()
    plt.show()
    
    print("Visualized middle Z-slices of first 3 frames")
else:
    print("No sample data available to visualize")

## 4. Data Statistics

Analyze intensity statistics across the dataset.

In [ ]:
if sample_data:
    # Collect statistics
    stats = {
        'frame': [],
        'min': [],
        'max': [],
        'mean': [],
        'std': [],
        'median': []
    }
    
    for frame_index, image in sample_data:
        stats['frame'].append(frame_index)
        stats['min'].append(float(image.min()))
        stats['max'].append(float(image.max()))
        stats['mean'].append(float(image.mean()))
        stats['std'].append(float(image.std()))
        stats['median'].append(float(np.median(image)))
    
    # Plot intensity statistics
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    axes[0, 0].plot(stats['frame'], stats['mean'], 'b-', marker='o', label='Mean')
    axes[0, 0].fill_between(stats['frame'], 
                             np.array(stats['mean']) - np.array(stats['std']),
                             np.array(stats['mean']) + np.array(stats['std']),
                             alpha=0.3)
    axes[0, 0].set_xlabel('Frame')
    axes[0, 0].set_ylabel('Intensity')
    axes[0, 0].set_title('Mean Intensity ± Std Dev')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(stats['frame'], stats['min'], 'g-', marker='s', label='Min')
    axes[0, 1].plot(stats['frame'], stats['max'], 'r-', marker='^', label='Max')
    axes[0, 1].set_xlabel('Frame')
    axes[0, 1].set_ylabel('Intensity')
    axes[0, 1].set_title('Min/Max Intensity')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].hist([img.flatten() for _, img in sample_data[:5]], bins=50, alpha=0.7)
    axes[1, 0].set_xlabel('Intensity')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Intensity Distribution (First 5 Frames)')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].plot(stats['frame'], stats['std'], 'm-', marker='d')
    axes[1, 1].set_xlabel('Frame')
    axes[1, 1].set_ylabel('Standard Deviation')
    axes[1, 1].set_title('Intensity Std Dev')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Statistics Summary:")
    print(f"  Mean intensity: {np.mean(stats['mean']):.2f}")
    print(f"  Std intensity: {np.mean(stats['std']):.2f}")
    print(f"  Min range: {np.min(stats['min']):.2f} - {np.max(stats['min']):.2f}")
    print(f"  Max range: {np.min(stats['max']):.2f} - {np.max(stats['max']):.2f}")